In [ ]:
import datetime
import pandas as pd
import plotly.graph_objects as go
from data_manager import DataMaster 

try:
    dm = DataMaster(raw_data)
    print("✅ DataMaster réinitialisé avec succès.")
except NameError:
    # Sinon, on recharge tout proprement
    print("🔄 Chargement initial des données...")
    from data_loader import load_period
    raw_data = load_period(start_year=2026, start_month=1)
    dm = DataMaster(raw_data)
    print("✅ Données rechargées et DataMaster prêt.")

start_date = datetime.date(2026, 1, 5)
days_to_show = []
current_date = start_date
while len(days_to_show) < 5:
    if current_date.weekday() < 5:
        days_to_show.append(current_date)
    current_date += datetime.timedelta(days=1)

for target_day in days_to_show:
    try:
        history, session, symbol = dm.get_market_context(target_day)
        
        if session is None or session.empty:
            print(f"⚠️ Aucune donnée pour le {target_day}")
            continue

        levels = dm.calculate_ict_levels(history, session)

        # Transformation H1
        session_h1 = session.resample('1h').agg({
            'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'
        }).dropna()

        fig = go.Figure(data=[go.Candlestick(
            x=session_h1.index,
            open=session_h1['open'], high=session_h1['high'],
            low=session_h1['low'], close=session_h1['close'],
            name=f"{symbol} H1"
        )])
        
        # Lignes ICT
        fig.add_hline(y=levels['mid'], line_color="cyan", line_width=2, 
                      annotation_text=f"Midnight Open: {levels['mid']}")
        
        if levels['pdh']:
            fig.add_hline(y=levels['pdh'], line_color="red", line_dash="dash", 
                          annotation_text=f"PDH: {levels['pdh']}")
        
        if levels['pdl']:
            fig.add_hline(y=levels['pdl'], line_color="green", line_dash="dash", 
                          annotation_text=f"PDL: {levels['pdl']}")

        fig.update_layout(
            title=f"NQ H1 - {target_day}",
            template="plotly_dark",
            xaxis_rangeslider_visible=False,
            hovermode="x unified",
            height=600
        )
        
        fig.show()
        
    except Exception as e:
        print(f"❌ Erreur le {target_day}: {e}")

In [1]:
import datetime
import pandas as pd
import plotly.graph_objects as go
from data_manager import DataMaster 
from data_loader import load_period

raw_data = load_period(
    start_year=2025, start_month=12, 
    end_year=2026, end_month=1
)
dm = DataMaster(raw_data)

start_date = datetime.date(2026, 1, 5)
days_to_show = [start_date + datetime.timedelta(days=i) for i in range(5)]

Target: 12/2025 to 1/2026
📥 Loading: MNQ_2025_12.pkl
📥 Loading: MNQ_2026_01.pkl
✅ Terminé : 82356 lignes chargées.


In [2]:
# --- Rendu Notebook Style "Engineering Report" ---

for target_day in days_to_show:
    history, session, symbol = dm.get_market_context(target_day)
    if session is None or session.empty: continue

    levels = dm.calculate_ict_levels(history, session)
    signals = dm.detect_signals(session, levels)
    po3 = dm.analyze_po3(history, session)

    print(f"\n{'═'*85}")
    print(f" REPORT ID: {symbol} | DATE: {target_day} ")
    print(f"{'═'*85}")

    # --- SECTION 1: KEY PIVOTS (HTF) ---
    print(f" [HTF PIVOTS]")
    print(f" ├─ Midnight Open : {levels.get('mid', 'N/A')}")
    print(f" ├─ Prev. Day High: {levels.get('pdh', 'N/A')}")
    print(f" └─ Prev. Day Low : {levels.get('pdl', 'N/A')}")
    print(f"{'-'*85}")

# --- SECTION 2: MAJOR LIQUIDITY LEVELS (DÉTAILLÉ) ---
    m_highs = levels.get('major_highs', [])
    m_lows = levels.get('major_lows', [])
    
    print(f" [MAJOR HIGHS H1 - BSL]")
    if m_highs:
        # On les affiche tous pour vérification
        print(f" ├─ Levels: {', '.join(map(str, m_highs))}")
    else:
        print(" ├─ No Major Highs found")

    print(f" [MAJOR LOWS H1 - SSL]")
    if m_lows:
        print(f" ├─ Levels: {', '.join(map(str, m_lows))}")
    else:
        print(" ├─ No Major Lows found")
    print(f"{'-'*85}")

    # --- SECTION 3: SIGNAL EXECUTION ---
    print(f" [EXECUTION LOGS]")
    if not signals:
        print("    >> No valid 'Grade A' setup detected in this window.")
    else:
        for s in signals:
            ote = s.get('ote_zone', {})
            print(f"    TIME: {s['time'].strftime('%H:%M')} | {s['type']}")
            print(f"    ├─ SOURCE LVL : {s['level_hit']}")
            print(f"    ├─ FVG GAP    : {s['fvg_gap']}")
            print(f"    ├─ OTE ENTRY  : {ote.get('entry_618')} - {ote.get('entry_786')}")
            print(f"    └─ RISK/STOP  : {ote.get('stop_loss')}")
            print("")

print(f"{'═'*85}\n")


═════════════════════════════════════════════════════════════════════════════════════
 REPORT ID: MNQH6 | DATE: 2026-01-05 
═════════════════════════════════════════════════════════════════════════════════════
 [HTF PIVOTS]
 ├─ Midnight Open : 25487.5
 ├─ Prev. Day High: 25491.5
 └─ Prev. Day Low : 25406.0
-------------------------------------------------------------------------------------
 [LIQUIDITY]
 ├─ Major Highs (BSL): 7 levels identified [26123.75, 26073.25]...
 └─ Major Lows  (SSL): 7 levels identified [24887.75, 24953.75]...
-------------------------------------------------------------------------------------
 [EXECUTION LOGS]
    >> No valid 'Grade A' setup detected in this window.

═════════════════════════════════════════════════════════════════════════════════════
 REPORT ID: MNQH6 | DATE: 2026-01-06 
═════════════════════════════════════════════════════════════════════════════════════
 [HTF PIVOTS]
 ├─ Midnight Open : 25646.75
 ├─ Prev. Day High: 25708.0
 └─ Prev. Day L